In [5]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
response = model.invoke("Why do parrots talk?")

response

AIMessage(content="Parrots are known for their remarkable ability to mimic human speech and other sounds, but why do they do it? The answer lies in their evolution, social behavior, and communication needs.\n\nIn the wild, parrots use vocalizations to communicate with each other, and mimicking sounds is an essential part of their language. They use vocalizations to:\n\n1. **Establish territory and identity**: By mimicking other birds, animals, or even man-made sounds, parrots can advertise their presence and warn others to stay away from their territory.\n2. **Attract a mate**: Male parrots will often mimic attractive sounds to impress potential mates and showcase their vocal abilities.\n3. **Warn of predators**: Parrots will alert their flock to potential threats by mimicking the calls of predators, such as hawks or snakes.\n4. **Maintain social bonds**: Parrots will often engage in vocal play, mimicking each other's sounds to strengthen social bonds and reinforce their relationships.

In [7]:
from langchain.tools import tool

@tool

def get_weather(location:str)->str:
    """Get the weather at a location."""
    return f"It is sunny in {location}."


model_with_tool = model.bind_tools([get_weather])    



In [10]:
response = model_with_tool.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'vy69en9pz', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.04432969, 'completion_tokens_details': None, 'prompt_time': 0.011254887, 'prompt_tokens_details': None, 'queue_time': 0.710104514, 'total_time': 0.055584577}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ec69d-0d0b-7e11-83a0-3dafe278a9e5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'vy69en9pz', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}
Tool:get_weather
Args:{'location': 'New York'}


### Tool Execution Loops

In [18]:
# Step1 = Model generates tool calls

messages = [{"role":"user","content":"What is the weather like in New York?'"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

#Step2 = Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step3 = Pass the result back to model for final response
final_response = model.invoke(messages)
print(final_response.content)    


New York's weather varies throughout the year. Here's a general overview:

1. **Spring (March to May)**: Mild temperatures, with average highs around 58°F (14°C) in March and 77°F (25°C) in May. Expect some rain showers and occasional warm spells.
2. **Summer (June to August)**: Hot and humid, with average highs often reaching the mid-80s to low 90s (29°C to 32°C). Summer is the wettest season, with occasional thunderstorms.
3. **Autumn (September to November)**: Comfortable temperatures, with average highs ranging from 64°F (18°C) in September to 47°F (8°C) in November. The fall foliage is a popular attraction during this time.
4. **Winter (December to February)**: Cold and snowy, with average temperatures ranging from 25°F (-4°C) in January (the coldest month) to 38°F (3°C) in December. Expect occasional cold snaps and snowstorms.

Keep in mind that these are general temperature ranges, and actual weather conditions can vary from year to year. If you're planning a trip to New York, i

In [19]:
messages

[{'role': 'user', 'content': "What is the weather like in New York?'"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jt2n3whcr', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.042283924, 'completion_tokens_details': None, 'prompt_time': 0.023628367, 'prompt_tokens_details': None, 'queue_time': 0.050384833, 'total_time': 0.065912291}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec6ad-7387-70c2-bbe8-182a28794b06-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'jt2n3whcr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}),
 ToolMessage(